# 01 — FastAPI CRUD API Demo (Document Uploader Service)

This notebook builds a **minimal FastAPI CRUD API** for the `documents` resource described in
chapter 02 (`02-building-the-service-with-fastapi.md`), backed by a real **SQLAlchemy** model against
an in-memory SQLite database, then exercises it using FastAPI's built-in **`TestClient`**
(`fastapi.testclient.TestClient`) — no running server process, no network calls, fully offline and
reproducible.

This mirrors the real service's shape (Pydantic request/response models, dependency-injected DB
session, CRUD path operations, correct status codes, soft-delete) and additionally demonstrates the
**RBAC** pattern from chapter 06 — using a mocked/decoded fake bearer token instead of a real Azure AD
call, since this notebook is meant to run offline with no MSAL or Azure AD dependency.

**What this demonstrates, matching chapters 01, 02, and 06's concepts:**
- A `Document` SQLAlchemy declarative model, `sessionmaker`, and `Depends(get_db)` session injection
- A `documents` router with full CRUD (`POST`, `GET` list + detail, `PUT`, `DELETE`)
- Pydantic request/response models — including a deliberately invalid payload triggering a `422`
- Correct status codes (`201`, `200`, `404`, `204`, `400`, `422`, `401`, `403`)
- A mocked RBAC dependency enforcing a role check on `DELETE`, showing both a `200`/`204` (correct
  role) and a `403` (wrong role) outcome — **no real MSAL/Azure AD call is made anywhere in this
  notebook**; the "token" is a plain string this notebook decodes itself, purely to demonstrate the
  `require_role(...)` dependency pattern in isolation.


In [1]:
import io
from datetime import datetime, timezone
from typing import Optional

from fastapi import FastAPI, APIRouter, Depends, HTTPException, UploadFile, File, status
from fastapi.security import HTTPAuthorizationCredentials, HTTPBearer
from fastapi.testclient import TestClient
from pydantic import BaseModel, Field

from sqlalchemy import create_engine, String, BigInteger, Integer, Boolean, DateTime, select
from sqlalchemy.orm import DeclarativeBase, Mapped, mapped_column, sessionmaker, Session
from sqlalchemy.pool import StaticPool

print("FastAPI + SQLAlchemy + TestClient demo — no server process, no external services required.")


FastAPI + SQLAlchemy + TestClient demo — no server process, no external services required.


C:\W\Interview Prep\CV Analysis\Interview-Prep\.venv\Lib\site-packages\fastapi\testclient.py:1: StarletteDeprecationWarning: Using `httpx` with `starlette.testclient` is deprecated; install `httpx2` instead.
  from starlette.testclient import TestClient as TestClient  # noqa


## 1. The SQLAlchemy model and session factory

Same shape as chapter 02/05's `Document` model (typed declarative columns, audit columns, soft-delete),
backed here by an **in-memory SQLite** database instead of Azure SQL. `StaticPool` plus
`check_same_thread=False` keeps the same in-memory database alive across the multiple `Session`s that
`Depends(get_db)` will create per-request — without it, SQLite's default in-memory behavior would give
every new connection an empty, disconnected database.


In [2]:
class Base(DeclarativeBase):
    pass


class Document(Base):
    __tablename__ = "documents"

    # NOTE: Azure SQL's actual column type is BIGINT (chapter 05) — this demo declares the primary
    # key as plain Integer instead, because SQLite only wires up autoincrement-on-insert behavior for
    # its INTEGER PRIMARY KEY rowid alias, not for a BIGINT-typed column. The ORM model shape (typed
    # declarative columns, audit columns, soft-delete) is otherwise identical to the production model.
    id: Mapped[int] = mapped_column(Integer, primary_key=True, autoincrement=True)
    filename: Mapped[str] = mapped_column(String(260), nullable=False)
    size_bytes: Mapped[int] = mapped_column(BigInteger, nullable=False)
    status: Mapped[str] = mapped_column(String(32), nullable=False, default="uploaded")

    created_by: Mapped[str] = mapped_column(String(128), nullable=False)
    created_at: Mapped[datetime] = mapped_column(DateTime, default=lambda: datetime.now(timezone.utc))
    updated_at: Mapped[datetime] = mapped_column(DateTime, default=lambda: datetime.now(timezone.utc),
                                                  onupdate=lambda: datetime.now(timezone.utc))

    is_deleted: Mapped[bool] = mapped_column(Boolean, nullable=False, default=False)
    deleted_at: Mapped[Optional[datetime]] = mapped_column(DateTime, nullable=True)
    deleted_by: Mapped[Optional[str]] = mapped_column(String(128), nullable=True)


engine = create_engine(
    "sqlite:///:memory:",
    connect_args={"check_same_thread": False},
    poolclass=StaticPool,
)
SessionLocal = sessionmaker(bind=engine, autoflush=False, expire_on_commit=False)
Base.metadata.create_all(engine)

print("Schema created via SQLAlchemy's declarative Base.metadata.create_all().")


Schema created via SQLAlchemy's declarative Base.metadata.create_all().


## 2. Pydantic request/response models

`DocumentUpdate` is the request shape used by `PUT`; `DocumentOut` is the response shape, read straight
off the SQLAlchemy object via `model_config = {"from_attributes": True}`. FastAPI validates every
request against these models (or the individual `Form`/`File` parameters below) before the path
operation body ever runs.


In [3]:
class DocumentUpdate(BaseModel):
    status: str = Field(min_length=1)


class DocumentOut(BaseModel):
    id: int
    filename: str
    status: str
    size_bytes: int
    created_by: str
    is_deleted: bool
    created_at: datetime
    updated_at: datetime

    model_config = {"from_attributes": True}


ALLOWED_STATUSES = {"uploaded", "processing", "ready", "failed"}
print("Pydantic models defined: DocumentUpdate (request), DocumentOut (response).")


Pydantic models defined: DocumentUpdate (request), DocumentOut (response).


## 3. The `get_db` dependency

Same generator-dependency pattern as chapter 02: yield a `Session`, close it after the response is
sent, regardless of success or failure.


In [4]:
def get_db():
    db = SessionLocal()
    try:
        yield db
    finally:
        db.close()


## 4. RBAC dependency — mocked token decoding, no real Azure AD/MSAL call

Chapter 06 validates a real Azure AD-issued JWT against Azure AD's JWKS endpoint. This notebook runs
fully offline, so `decode_fake_token` below is a **plain string parser that simulates having already
decoded a token's claims** — it does no cryptographic signature verification and talks to no network
endpoint. Its only job is to let this notebook demonstrate the shape of the `require_role(...)`
dependency (chapter 06) in isolation, using a bearer "token" of the form
`"role:<Role1,Role2>;user:<email>"` that this notebook itself constructs for the demo.

**In the real service, `get_current_user` instead validates a real Azure AD JWT's signature against the
tenant's JWKS endpoint and reads `roles`/`oid`/`preferred_username` out of validated claims — see
chapter 06 for that implementation.**


In [5]:
bearer_scheme = HTTPBearer()


def decode_fake_token(token: str) -> dict:
    """NOT real JWT validation. Simulates already-decoded claims for this offline demo only."""
    claims = {"role": "", "user": "unknown"}
    for part in token.split(";"):
        if ":" not in part:
            continue
        key, _, value = part.partition(":")
        claims[key] = value
    return claims


def get_current_user(credentials: HTTPAuthorizationCredentials = Depends(bearer_scheme)) -> dict:
    claims = decode_fake_token(credentials.credentials)
    roles = [r for r in claims.get("role", "").split(",") if r]
    return {"email": claims.get("user", "unknown"), "roles": roles}


def require_role(required_role: str):
    def _check(current_user: dict = Depends(get_current_user)) -> None:
        if required_role not in current_user["roles"]:
            raise HTTPException(
                status_code=status.HTTP_403_FORBIDDEN,
                detail=f"missing required role: {required_role}",
            )
    return _check


print("Mocked RBAC dependencies defined: get_current_user, require_role (offline demo only).")


Mocked RBAC dependencies defined: get_current_user, require_role (offline demo only).


## 5. The `documents` router (CRUD path operations)

Same five path operations as chapter 02's worked example. `DELETE` is the one endpoint gated by
`Depends(require_role("DocumentUploader.Write"))` — every other endpoint only requires a valid
(decodable) bearer token, no specific role.


In [6]:
router = APIRouter(prefix="/v1/documents", tags=["documents"])


@router.post("", response_model=DocumentOut, status_code=201)
async def create_document(
    file: UploadFile = File(...),
    db: Session = Depends(get_db),
    current_user: dict = Depends(get_current_user),
):
    if not file.filename:
        raise HTTPException(status_code=400, detail="no file selected")
    content = await file.read()
    doc = Document(
        filename=file.filename,
        size_bytes=len(content),
        status="uploaded",
        created_by=current_user["email"],
    )
    db.add(doc)
    db.commit()
    db.refresh(doc)
    return doc


@router.get("", response_model=list[DocumentOut])
async def list_documents(
    status_filter: Optional[str] = None,
    page: int = 1,
    page_size: int = 10,
    db: Session = Depends(get_db),
    current_user: dict = Depends(get_current_user),
):
    stmt = select(Document).where(Document.is_deleted == False)  # noqa: E712
    if status_filter:
        stmt = stmt.where(Document.status == status_filter)
    stmt = stmt.order_by(Document.created_at.desc())
    stmt = stmt.offset((page - 1) * page_size).limit(page_size)
    return db.scalars(stmt).all()


@router.get("/{doc_id}", response_model=DocumentOut)
async def get_document(doc_id: int, db: Session = Depends(get_db),
                        current_user: dict = Depends(get_current_user)):
    doc = db.scalar(select(Document).where(Document.id == doc_id, Document.is_deleted == False))  # noqa: E712
    if doc is None:
        raise HTTPException(status_code=404, detail="not found")
    return doc


@router.put("/{doc_id}", response_model=DocumentOut)
async def update_document(doc_id: int, body: DocumentUpdate, db: Session = Depends(get_db),
                           current_user: dict = Depends(get_current_user)):
    doc = db.scalar(select(Document).where(Document.id == doc_id, Document.is_deleted == False))  # noqa: E712
    if doc is None:
        raise HTTPException(status_code=404, detail="not found")
    if body.status not in ALLOWED_STATUSES:
        raise HTTPException(status_code=400, detail=f"status must be one of {sorted(ALLOWED_STATUSES)}")
    doc.status = body.status
    db.commit()
    db.refresh(doc)
    return doc


@router.delete("/{doc_id}", status_code=204)
async def delete_document(
    doc_id: int,
    db: Session = Depends(get_db),
    current_user: dict = Depends(get_current_user),
    _: None = Depends(require_role("DocumentUploader.Write")),
):
    doc = db.scalar(select(Document).where(Document.id == doc_id, Document.is_deleted == False))  # noqa: E712
    if doc is None:
        raise HTTPException(status_code=404, detail="not found")
    doc.is_deleted = True
    doc.deleted_by = current_user["email"]
    doc.deleted_at = datetime.now(timezone.utc)
    db.commit()


app = FastAPI(title="Document Uploader Service (demo)")
app.include_router(router)
client = TestClient(app)
print("FastAPI app assembled, TestClient ready — no server process running.")


FastAPI app assembled, TestClient ready — no server process running.


## 6. Exercising `POST /v1/documents` (Create)

A valid multipart upload with a bearer token that carries no special role (read-only access is enough
to create/list/read/update in this demo — only `DELETE` is role-gated).


In [7]:
READ_TOKEN = "Bearer role:DocumentUploader.Read;user:bob@hsbc.com"
WRITE_TOKEN = "Bearer role:DocumentUploader.Write;user:alice@hsbc.com"

response = client.post(
    "/v1/documents",
    files={"file": ("q3-policy.pdf", io.BytesIO(b"Q3 compliance policy contents..."), "application/pdf")},
    headers={"Authorization": READ_TOKEN},
)
print("Status:", response.status_code)
print("Body:", response.json())
first_doc_id = response.json()["id"]


Status: 201
Body: {'id': 1, 'filename': 'q3-policy.pdf', 'status': 'uploaded', 'size_bytes': 32, 'created_by': 'bob@hsbc.com', 'is_deleted': False, 'created_at': '2026-07-15T15:39:13.335775', 'updated_at': '2026-07-15T15:39:13.335778'}


## 7. Pydantic validation catching a bad payload — `422`

`PUT /v1/documents/{id}` expects a `DocumentUpdate` body with a non-empty `status` string. Sending a
body that doesn't match the model's shape at all (missing the required field) is rejected by FastAPI's
Pydantic-based validation **before** the route function runs, with a structured `422` response —
distinct from the `400` this route raises itself for a status value that's present but not one of the
allowed transitions.


In [8]:
# Missing the required `status` field entirely -> FastAPI/Pydantic rejects with 422
bad_shape_response = client.put(
    f"/v1/documents/{first_doc_id}",
    json={"not_status": "ready"},
    headers={"Authorization": READ_TOKEN},
)
print("Malformed body -> status:", bad_shape_response.status_code, "(422 = Pydantic validation failure)")
print(bad_shape_response.json())

# Well-formed body, but not an allowed status value -> the route's own business-rule check returns 400
bad_value_response = client.put(
    f"/v1/documents/{first_doc_id}",
    json={"status": "not-a-real-status"},
    headers={"Authorization": READ_TOKEN},
)
print("\nWell-formed but invalid status value -> status:", bad_value_response.status_code,
      "(400 = business rule, not a shape/validation problem)")
print(bad_value_response.json())


Malformed body -> status: 422 (422 = Pydantic validation failure)
{'detail': [{'type': 'missing', 'loc': ['body', 'status'], 'msg': 'Field required', 'input': {'not_status': 'ready'}}]}

Well-formed but invalid status value -> status: 400 (400 = business rule, not a shape/validation problem)
{'detail': "status must be one of ['failed', 'processing', 'ready', 'uploaded']"}


## 8. Exercising `GET` (list + detail) and `PUT` (valid update)


In [9]:
client.post("/v1/documents",
            files={"file": ("onboarding.docx", io.BytesIO(b"onboarding guide"),
                             "application/vnd.openxmlformats-officedocument.wordprocessingml.document")},
            headers={"Authorization": READ_TOKEN})
client.post("/v1/documents",
            files={"file": ("runbook.pdf", io.BytesIO(b"runbook v2"), "application/pdf")},
            headers={"Authorization": READ_TOKEN})

list_response = client.get("/v1/documents?page=1&page_size=2", headers={"Authorization": READ_TOKEN})
print("List status:", list_response.status_code)
for d in list_response.json():
    print(" -", d["id"], d["filename"], d["status"])

update_response = client.put(f"/v1/documents/{first_doc_id}", json={"status": "ready"},
                              headers={"Authorization": READ_TOKEN})
print("\nUpdate status:", update_response.status_code)
print(update_response.json())


List status: 200
 - 3 runbook.pdf uploaded
 - 2 onboarding.docx uploaded

Update status: 200
{'id': 1, 'filename': 'q3-policy.pdf', 'status': 'ready', 'size_bytes': 32, 'created_by': 'bob@hsbc.com', 'is_deleted': False, 'created_at': '2026-07-15T15:39:13.335775', 'updated_at': '2026-07-15T15:39:13.377787'}


## 9. RBAC demo on `DELETE` — wrong role (`403`) vs. correct role (`200`/`204`)

This is the core RBAC demonstration from chapter 06's `require_role(...)` pattern, exercised end to
end with the mocked token decoder from section 4. **No real Azure AD or MSAL call happens anywhere in
this cell** — `READ_TOKEN`/`WRITE_TOKEN` are plain strings this notebook constructs itself, decoded by
`decode_fake_token`, purely to prove the dependency's pass/fail behavior in isolation.


In [10]:
# Wrong role: DocumentUploader.Read cannot delete
forbidden_response = client.delete(f"/v1/documents/{first_doc_id}", headers={"Authorization": READ_TOKEN})
print("DELETE with Read-only role -> status:", forbidden_response.status_code, "(403 = missing role)")
print(forbidden_response.json())

# No token at all
no_auth_response = client.delete(f"/v1/documents/{first_doc_id}")
print("\nDELETE with no Authorization header -> status:", no_auth_response.status_code,
      "(401/403 depending on the security scheme's no-credentials behavior)")

# Correct role: DocumentUploader.Write can delete
allowed_response = client.delete(f"/v1/documents/{first_doc_id}", headers={"Authorization": WRITE_TOKEN})
print("\nDELETE with Write role -> status:", allowed_response.status_code, "(204 = soft-deleted)")

after_delete = client.get(f"/v1/documents/{first_doc_id}", headers={"Authorization": READ_TOKEN})
print("GET after delete -> status:", after_delete.status_code, "(404, looks deleted to callers)")


DELETE with Read-only role -> status: 403 (403 = missing role)
{'detail': 'missing required role: DocumentUploader.Write'}

DELETE with no Authorization header -> status: 401 (401/403 depending on the security scheme's no-credentials behavior)

DELETE with Write role -> status: 204 (204 = soft-deleted)
GET after delete -> status: 404 (404, looks deleted to callers)


## 10. Not-found handling


In [11]:
missing_response = client.get("/v1/documents/99999", headers={"Authorization": READ_TOKEN})
print("Status:", missing_response.status_code)
print("Body:", missing_response.json())


Status: 404
Body: {'detail': 'not found'}


## Summary

| Verb | Endpoint | Status codes demonstrated |
|---|---|---|
| POST | `/v1/documents` | `201` (create) |
| PUT | `/v1/documents/{id}` | `422` (malformed body — Pydantic), `400` (invalid status value), `200` (valid update) |
| GET | `/v1/documents` | `200` (list + pagination) |
| GET | `/v1/documents/{id}` | `200` (found), `404` (not found / soft-deleted) |
| DELETE | `/v1/documents/{id}` | `403` (wrong RBAC role), `401`/`403` (no token), `204` (correct role, soft-deleted) |

The RBAC check (`Depends(require_role("DocumentUploader.Write"))`) and the SQLAlchemy-backed `Document`
model are the same *pattern* used in the real service (chapters 02 and 06) — the only things swapped out
for this notebook are the token source (a plain string here vs. a real Azure AD-issued, MSAL-acquired
JWT in production) and the database engine (in-memory SQLite here vs. Azure SQL in production). See
`notebooks/02_sql_data_layer_demo.ipynb` for the SQLAlchemy data-layer patterns (soft-delete, tenant
scoping, pagination) in more depth.
